In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
GRN_ROOT = './data'

In [ ]:
GRN = pd.read_csv(f'{GRN_ROOT}/100per_act_inferred_GRN_norm.csv', index_col = 0)

In [ ]:
GRN['Reg_name'] = np.where(GRN['Reg'] == -1, 'Inhibitory', 'Excitatory')

In [ ]:
graph = nx.from_pandas_edgelist(GRN, source = 'TF', target='Gene', edge_attr='Reg_name', create_using = nx.DiGraph)

# GRN properties examinations

In [ ]:
#get list of gene names that are TFs only, TGs only, and both
DATA_ROOT = '../../data'
network = pd.read_csv(f'{DATA_ROOT}/Full data files/network(full).tsv', sep='\t')
TFs = set(GRN['TF'])
TGs = set(GRN['Gene'])
TF_TGs = TFs & TGs
#mkae TFs set the TFs that are not also TGs
TFs = TFs - TF_TGs
TGs = TGs - TF_TGs
print(f'There are {len(TFs)} TFs and {len(TGs)} TGs. {len(TF_TGs)} TFs are also TGs')

In [ ]:
in_degree_dict = {}
out_degree_dict = {}

for node in graph.nodes():
    in_degree_dict[node] = graph.in_degree(node)
    out_degree_dict[node] = graph.out_degree(node)

In [ ]:
TF_in_degrees = {k: in_degree_dict[k] for k in TFs}
TF_out_degrees = {k: out_degree_dict[k] for k in TFs}

In [ ]:
TG_in_degrees = {k: in_degree_dict[k] for k in TGs}
TG_out_degrees = {k: out_degree_dict[k] for k in TGs}

In [ ]:
TF_TG_in_degrees = {k: in_degree_dict[k] for k in TF_TGs}
TF_TG_out_degrees = {k: out_degree_dict[k] for k in TF_TGs}

In [ ]:
fig, ax = plt.subplots(ncols = 2, figsize = (15, 5))
ax[0].hist(TF_out_degrees.values(), bins = 20, color = '#b00000')
ax[0].set_title('TFs (Only)', fontsize = 15)
ax[1].hist(TF_TG_out_degrees.values(), bins = 20, color = '#6b00d6')
ax[1].set_title('TF and Target Genes', fontsize = 15)
ax[0].tick_params(axis='both', which='major', labelsize=15)
ax[1].tick_params(axis='both', which='major', labelsize=15)
fig.suptitle('Out Degree Distributions', fontsize = 20)
plt.savefig('./figures/OutDistributions_norm', dpi = 300, bbox_inches = 'tight')

In [ ]:
fig, ax = plt.subplots(ncols = 2, figsize = (15, 5))
ax[0].hist(TG_in_degrees.values(), bins = 20, color = '#03038f')
ax[0].set_title('Target Genes (Only)', fontsize = 15)
ax[1].hist(TF_TG_in_degrees.values(), bins = 20, color = '#6b00d6')
ax[1].set_title('TF and Target Genes', fontsize = 15)
ax[0].tick_params(axis='both', which='major', labelsize=15)
ax[1].tick_params(axis='both', which='major', labelsize=15)
fig.suptitle('In Degree Distributions', fontsize = 20)
plt.savefig('./figures/InDistributions_norm', dpi = 300, bbox_inches = 'tight')

In [ ]:
degree_centralities = nx.degree_centrality(graph)
TF_TG_centralities = {k: degree_centralities[k] for k in TF_TGs}

In [ ]:
node_betweeness = nx.betweenness_centrality(graph)
TF_TG_node_betweeness = {k: node_betweeness[k] for k in TF_TGs}

In [ ]:
TFTG_df = pd.DataFrame([TF_TG_in_degrees, TF_TG_out_degrees, TF_TG_centralities, TF_TG_node_betweeness], index = ['in degree', 'out degree', 'Centrality', 'Betweeness']).T

In [ ]:
fig, ax = plt.subplots(1)
sns.scatterplot(TFTG_df, x = 'in degree', y = 'out degree', hue = 'Centrality', s = 7, palette = 'icefire')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_ylabel('Out Degree', fontsize = 15)
ax.set_xlabel('In Degree', fontsize = 15)
ax.set_title('TF-TG Centrality', fontsize = 20)
ax.tick_params(axis='both', which='major', labelsize=15)
plt.savefig('./figures/InOutCentralityTFTG_norm', dpi = 300, bbox_inches = 'tight')

In [ ]:
fig, ax = plt.subplots(1)
sns.scatterplot(TFTG_df, x = 'in degree', y = 'out degree', hue = 'Betweeness', s = 7, palette = 'icefire')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_ylabel('Out Degree', fontsize = 15)
ax.set_xlabel('In Degree', fontsize = 15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.set_title('TF-TG Betweeness', fontsize = 20)
plt.savefig('./figures/InOutBetweenessTFTG_norm', dpi = 300, bbox_inches = 'tight')

# Single gene focused graph plotting

In [ ]:
gene = 'TP53'
#PIP4K2C

In [ ]:
subgene_graph_df = GRN[(GRN['Gene'] == gene) | (GRN['TF'] == gene)]
subgene_graph = nx.from_pandas_edgelist(subgene_graph_df, source = 'TF', target='Gene', edge_attr='Reg_name', create_using = nx.DiGraph)
subgene_graph_df

In [ ]:
reg_type = nx.get_edge_attributes(subgene_graph,'Reg_name').values()
reg_type

In [ ]:
#create a dictionary mapping reg type to colors
cmap = {
    'Inhibitory' : 'red',
    'Excitatory' : 'blue'
}

color_list = [cmap[reg_t] for reg_t in reg_type]

In [ ]:
gene in subgene_graph.nodes()

In [ ]:
fig, ax = plt.subplots(figsize=(30,30))

layout = nx.arf_layout(subgene_graph)
nx.draw(subgene_graph,
        layout,
        edge_color= color_list,
        with_labels=True,
        node_size = 10)
plt.savefig(f"{gene}_net", dpi = 300)